# Chapter 2: Audio for ML engineers

This notebook runs every piece of code in the chapter, from the saved results, in order. Read the chapter alongside it: [Audio for ML engineers](https://books.sugiv.fyi/voice_applications/audio-for-ml.html).


In [ ]:
# Run this cell first. In Google Colab it fetches the companion repo; anywhere else, run the notebook
# from a clone of it. Either way it moves into the book's folder and installs what the chapter needs.
import os
import pathlib
import sys

if "google.colab" in sys.modules and not pathlib.Path("/content/voices-of-the-future").exists():
    !git clone -q --depth 1 https://github.com/sugix/voices-of-the-future /content/voices-of-the-future
candidates = [pathlib.Path.cwd(), *pathlib.Path.cwd().parents, pathlib.Path("/content/voices-of-the-future")]
root = next(p for p in candidates if (p / "voice" / "sim").is_dir())
os.chdir(root / "voice")
%pip install -q -r ../requirements/results.txt

## Sound is a list of numbers

In [ ]:
import json
import struct
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio, display
from scipy import signal


def read_wav(path):
    """Parse a WAV file by hand: a few labelled chunks of header, then the raw sample bytes."""
    data = Path(path).read_bytes()
    assert data[:4] == b"RIFF" and data[8:12] == b"WAVE", "not a WAV file"
    pos, info = 12, {"bytes": len(data)}
    while pos + 8 <= len(data):
        chunk, size = data[pos:pos + 4], struct.unpack("<I", data[pos + 4:pos + 8])[0]
        body = data[pos + 8:pos + 8 + size]
        if chunk == b"fmt ":
            code, channels, sr, byte_rate, _, bits = struct.unpack("<HHIIHH", body[:16])
            info.update(code=code, channels=channels, sr=sr, byte_rate=byte_rate, bits=bits)
        elif chunk == b"data":
            info["raw"] = body
        pos += 8 + size + (size & 1)  # chunks are padded to an even length
    return info


for path in ["data/ch2/c01_status_24k.wav", "data/calls/phone/c01_status.wav"]:
    w = read_wav(path)
    print(f"{path:34} format {w['code']}, {w['channels']} channel, {w['sr']:,} Hz, {w['bits']} bits, "
          f"{w['byte_rate'] * 8 / 1000:.0f} kbit/s, {len(w['raw']) / w['byte_rate']:.2f} s, {w['bytes']:,} bytes")

In [ ]:
def mulaw_decode(raw):
    """G.711 mu-law: each byte is a sign bit, a 3-bit exponent and a 4-bit mantissa, stored inverted."""
    u = ~np.frombuffer(raw, dtype=np.uint8)
    exponent, mantissa = (u >> 4) & 7, (u & 15).astype(np.int32)
    magnitude = (((mantissa << 3) + 0x84) << exponent) - 0x84
    return np.where(u & 0x80, -magnitude, magnitude) / 32768


def load(path):
    """Samples as float32 in [-1, 1], and the sample rate."""
    w = read_wav(path)
    if w["code"] == 1 and w["bits"] == 16:
        x = np.frombuffer(w["raw"], dtype="<i2") / 32768
    elif w["code"] == 7:
        x = mulaw_decode(w["raw"])
    else:
        raise ValueError(f"WAV format {w['code']} is not handled here")
    if w["channels"] > 1:
        x = x.reshape(-1, w["channels"]).mean(axis=1)
    return x.astype(np.float32), w["sr"]


clean, clean_sr = load("data/ch2/c01_status_24k.wav")
phone, phone_sr = load("data/calls/phone/c01_status.wav")
i = int(1.3 * phone_sr)  # 1.3 seconds in, mid-word
raw = read_wav("data/calls/phone/c01_status.wav")["raw"]
print("phone bytes  ", list(raw[i:i + 6]))
print("as samples   ", phone[i:i + 6].round(4))
print(f"clean: {len(clean):,} samples at {clean_sr:,} Hz; phone: {len(phone):,} samples at {phone_sr:,} Hz")
display(Audio(clean, rate=clean_sr), Audio(phone, rate=phone_sr))

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(9, 6))
for a, x, sr, name, c in [(ax[0], clean, clean_sr, "clean, 24 kHz", "C0"), (ax[1], phone, phone_sr, "phone, 8 kHz", "C1")]:
    t = np.arange(len(x)) / sr
    a.plot(t, x, lw=0.4, color=c)
    a.set(ylabel="amplitude", title=name)
    zoom = (t >= 0.42) & (t < 0.44)
    ax[2].plot(t[zoom] * 1000, x[zoom] / np.abs(x).max(), ".-", ms=3, lw=0.6, color=c, label=name)
ax[1].set_xlabel("seconds")
ax[2].set(xlabel="milliseconds", ylabel="scaled amplitude", title="20 ms of the first word: every dot is one sample")
ax[2].legend(loc="upper right", fontsize=8)
fig.tight_layout()

## What the phone line takes away

### Everything above 3.4 kHz

In [ ]:
f, p = signal.welch(clean, clean_sr, nperseg=1024)
above = p[f > 3400].sum() / p.sum()
frames = clean[: len(clean) // 480 * 480].reshape(-1, 480)  # 20 ms frames
spec = np.abs(np.fft.rfft(frames * np.hanning(480), axis=1)) ** 2
fb = np.fft.rfftfreq(480, 1 / clean_sr)
loud = (frames**2).mean(axis=1) > 1e-5
hi_share = spec[:, fb > 3400].sum(axis=1) / spec.sum(axis=1)
print(f"energy above 3.4 kHz: {above:.1%} of the call")
print(f"20 ms frames with sound: {loud.sum()}, of which {np.sum(loud & (hi_share > 0.5))} have most energy above 3.4 kHz")

fp, pp = signal.welch(phone, phone_sr, nperseg=512)
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.semilogy(f, p, lw=1, label="clean, 24 kHz")
ax.semilogy(fp, pp * (p.max() / pp.max()), lw=1, label="phone, 8 kHz (scaled)")
ax.axvspan(300, 3400, color="C2", alpha=0.1, label="phone band, 300 to 3,400 Hz")
ax.set(xlabel="frequency (Hz)", ylabel="power", title="Where the sound energy is", xlim=(0, 12000))
ax.legend(fontsize=8)
fig.tight_layout()

### Resampling: filter first, or you invent sounds

In [ ]:
sr = 24000
t = np.arange(2 * sr) / sr
chirp = 0.5 * signal.chirp(t, f0=100, t1=2, f1=11900)  # a tone sweeping up to 11.9 kHz
sliced = chirp[::3]  # 'downsample' to 8 kHz by keeping every third sample
proper = signal.resample_poly(chirp, 1, 3)  # filter first, then keep every third sample

tone = np.sin(2 * np.pi * 5000 * t)  # a 5 kHz tone cannot exist at 8 kHz
for name, x in [("every third sample", tone[::3]), ("resample_poly", signal.resample_poly(tone, 1, 3))]:
    mid = x[1000:-1000]  # skip the filter's start-up at each end
    spec = np.abs(np.fft.rfft(mid * np.hanning(len(mid))))
    freqs = np.fft.rfftfreq(len(mid), 1 / 8000)
    level = 20 * np.log10(np.sqrt(np.mean(mid**2)) / np.sqrt(0.5))  # relative to the original tone
    print(f"{name:20} strongest frequency {freqs[spec.argmax()]:5.0f} Hz, level {level:6.1f} dB")

fig, ax = plt.subplots(1, 2, figsize=(9, 3), sharey=True)
for a, x, name in [(ax[0], sliced, "every third sample"), (ax[1], proper, "resample_poly")]:
    a.specgram(x, Fs=8000, NFFT=256, noverlap=192, cmap="magma", vmin=-120)
    a.set(title=name, xlabel="seconds")
ax[0].set_ylabel("frequency (Hz)")
fig.tight_layout()
display(Audio(sliced, rate=8000), Audio(proper, rate=8000))

### μ-law: 8 bits that behave like 14

In [ ]:
MU = 255


def mulaw_8bit(x):
    """Compress, round to 8 bits, expand: the continuous form of G.711."""
    y = np.sign(x) * np.log1p(MU * np.abs(x)) / np.log1p(MU)
    y = np.round(y * 127) / 127
    return np.sign(y) * np.expm1(np.abs(y) * np.log1p(MU)) / MU


def linear_8bit(x):
    return np.round(x * 127) / 127


def snr_db(x, y):
    return 10 * np.log10(np.sum(x**2) / np.sum((x - y) ** 2))


speech = signal.resample_poly(clean, 1, 3)
speech = speech / np.abs(speech).max()
print("peak level   linear 8-bit   mu-law 8-bit")
for level in [0, -10, -20, -30, -40]:
    x = speech * 10 ** (level / 20)
    print(f"{level:6} dB   {snr_db(x, linear_8bit(x)):9.1f} dB   {snr_db(x, mulaw_8bit(x)):9.1f} dB")

### The whole phone line in one function

In [ ]:
PHONE_SR = 8_000


def to_phone(audio, sr, snr_db, rng):
    """Simulate a phone line: 8 kHz, 300-3400 Hz band, background noise."""
    down = signal.resample_poly(audio, PHONE_SR, sr)
    sos = signal.butter(4, [300, 3400], btype="bandpass", fs=PHONE_SR, output="sos")
    band = signal.sosfilt(sos, down)
    noise = rng.normal(0.0, 1.0, size=band.shape)
    p_sig = np.mean(band**2)
    noise *= np.sqrt(p_sig / (10 ** (snr_db / 10)) / np.mean(noise**2))
    out = band + noise
    return (0.9 * out / np.max(np.abs(out))).astype(np.float32)


mine = to_phone(clean, clean_sr, snr_db=25, rng=np.random.default_rng(0))
for name, x in [("my to_phone", mine), ("the book's phone file", phone)]:
    f2, p2 = signal.welch(x, PHONE_SR, nperseg=512)
    band = (f2 >= 300) & (f2 <= 3400)
    print(f"{name:22} {len(x):6} samples, {p2[band].sum() / p2.sum():.1%} of power inside 300-3,400 Hz")

## What a speech model actually sees

In [ ]:
def log_mel(x, sr=16000, n_fft=512, win=400, hop=160, n_mels=80):
    """The features most speech models read: 25 ms windows every 10 ms, 80 mel bands, log power."""
    frames = np.lib.stride_tricks.sliding_window_view(x, win)[::hop] * np.hanning(win)
    power = np.abs(np.fft.rfft(frames, n_fft)) ** 2
    mel = lambda hz: 2595 * np.log10(1 + hz / 700)
    hz = lambda m: 700 * (10 ** (m / 2595) - 1)
    edges = hz(np.linspace(mel(0), mel(sr / 2), n_mels + 2))
    bins = np.fft.rfftfreq(n_fft, 1 / sr)
    lo, mid, hi = edges[:-2, None], edges[1:-1, None], edges[2:, None]
    weights = np.maximum(0, np.minimum((bins - lo) / (mid - lo), (hi - bins) / (hi - mid)))
    return np.log(power @ weights.T + 1e-10), edges[1:-1]


clean16 = signal.resample_poly(clean, 2, 3)
phone16 = signal.resample_poly(phone, 2, 1)
M_clean, centres = log_mel(clean16)
M_phone, _ = log_mel(phone16)
top = centres > 3400
print(f"{M_clean.shape[0]} frames x {M_clean.shape[1]} bands; {top.sum()} bands sit above 3.4 kHz")
ratio = np.exp(M_phone[:, top]).sum() / np.exp(M_clean[:, top]).sum()
print(f"energy in those bands, phone against clean: {10 * np.log10(ratio):.0f} dB")
print(f"bands above 4 kHz: {np.sum(centres > 4000)}, and the phone version cannot fill any of them")

fig, ax = plt.subplots(1, 2, figsize=(9, 3.2), sharey=True)
for a, M, name in [(ax[0], M_clean, "clean"), (ax[1], M_phone, "phone")]:
    a.imshow(M.T, origin="lower", aspect="auto", cmap="magma", vmin=M_clean.max() - 18, vmax=M_clean.max(),
             extent=[0, M.shape[0] / 100, 0, 80])
    a.set(title=f"log-mel, {name}", xlabel="seconds")
ax[0].set_ylabel("mel band")
fig.tight_layout()

## Six ways to hand audio to a model, measured

In [ ]:
sys.path.insert(0, "sim")
from scoring import entity_hits, word_errors

refs = {f"halfwheel/{r['id']}": r for r in map(json.loads, open("data/calls/manifest.jsonl"))}
refs |= {f"librispeech/{r['id']}": r for r in map(json.loads, open("data/librispeech/manifest.jsonl"))}
cap = json.load(open("data/ch2/capture_ch2.json"))


def score(res, which):
    errs = [word_errors(refs[k]["written"], r["text"]) for k, r in res.items() if k.startswith(which)]
    return sum(e for e, _ in errs) / sum(n for _, n in errs)


print(f"{'what Parakeet-TDT was given':52} {'Halfwheel':>10} {'LibriSpeech':>12} {'entities':>9}")
for cond, res in cap["asr"].items():
    hits = [ok for k, r in res.items() if k.startswith("halfwheel") for _, ok in entity_hits(refs[k], r["text"])]
    print(f"{cond:52} {score(res, 'halfwheel'):10.1%} {score(res, 'librispeech'):12.1%} {sum(hits):5}/{len(hits)}")
print()
for cond, res in cap["asr"].items():
    print(f"{cond[:44]:44} {res['halfwheel/c01_status']['text']}")

## Is anyone speaking? Voice activity detection

In [ ]:
truth = json.load(open("data/ch2/vad_truth.json"))
stream, sr = load("data/ch2/vad_stream.wav")
FRAME = truth["frame"]
labels = np.array(truth["labels"])
frames = stream[: len(labels) * FRAME].reshape(-1, FRAME)
energy_db = 10 * np.log10(np.mean(frames**2, axis=1) + 1e-10)
kind = np.empty(len(labels), dtype=object)
for seg in truth["segments"]:
    a = round(seg["start_s"] * sr / FRAME)
    kind[a:a + round(seg["seconds"] * sr / FRAME)] = seg["kind"]


def energy_vad(db, margin=15):
    floor = np.percentile(db, 10)  # the quietest frames are the line noise
    return db > floor + margin


def report(name, speech):
    out = {"VAD": name, "speech heard": np.mean(speech[labels == 1])}
    for k in ["quiet line", "road noise", "hold music", "workshop clatter"]:
        out[k] = np.mean(speech[(labels == 0) & (kind == k)])
    return out


print(f"{'margin':>8} {'speech heard':>13}" + "".join(f"{k:>18}" for k in ["quiet line", "road noise", "hold music", "workshop clatter"]))
for margin in [10, 15, 20, 25, 30]:
    r = report("", energy_vad(energy_db, margin))
    print(f"{margin:5} dB {r['speech heard']:13.0%}" + "".join(f"{v:18.0%}" for k, v in list(r.items())[2:]))

In [ ]:
silero = np.array(cap["silero"])
results = [report("energy, 15 dB", energy_vad(energy_db)), report("Silero VAD", silero > 0.5)]
print(f"{'':16}" + "".join(f"{k:>18}" for k in results[0] if k != "VAD"))
for r in results:
    print(f"{r['VAD']:16}" + "".join(f"{v:18.0%}" for k, v in r.items() if k != "VAD"))

t = np.arange(len(labels)) * FRAME / sr
fig, ax = plt.subplots(2, 1, figsize=(9, 4.4), sharex=True)
ax[0].plot(np.arange(len(stream)) / sr, stream, lw=0.3, color="0.4")
for seg in truth["segments"]:
    if seg["kind"] != "speech":
        for a in ax:
            a.axvspan(seg["start_s"], seg["start_s"] + seg["seconds"], color="C3", alpha=0.12)
        ax[0].text(seg["start_s"] + 0.1, 0.8, seg["kind"], fontsize=7)
ax[0].set(ylabel="amplitude", title="Eight calls, and what came down the line between them (shaded)")
ax[1].fill_between(t, 0, energy_vad(energy_db), step="post", alpha=0.5, label="energy, 15 dB")
ax[1].fill_between(t, 0, -(silero > 0.5).astype(float), step="post", alpha=0.5, color="C2", label="Silero")
ax[1].set(yticks=[], xlabel="seconds", title="Where each detector says someone is speaking")
ax[1].legend(loc="lower right", fontsize=8)
fig.tight_layout()

## Rerun the models

Everything above used the saved results. The cells below rerun the models that produced them, on a GPU (in Colab:
Runtime, then Change runtime type, then T4 GPU). Each step builds the software it needs in its own environment the
first time, which takes a few minutes, then runs. Set `VOTF_QUICK=2` first for a two-item check that everything
works; leave it unset for a full rerun, which overwrites the saved results. On GPUs without bfloat16, such as the
T4, the scripts use float16 automatically.

In [ ]:
import os
os.environ["VOTF_QUICK"] = "2"  # remove this line for a full rerun

Halfwheel's calls: Kokoro reads them, Chapter 2's phone line carries them.

In [ ]:
!bash tools/stack.sh tts
!~/.votf/envs/tts/bin/python sim/halfwheel_calls.py

In [ ]:
!bash tools/stack.sh nemo
!~/.votf/envs/nemo/bin/python sim/capture_ch2.py